# Track 5 (Classical ML) · Linear & Logistic Regression

The two foundational supervised models, from scratch in NumPy. Linear regression predicts a number (matches the closed-form normal equation); logistic regression predicts a probability. Both train with the same gradient shape.

In [ ]:
"""
Track 5 (Classical ML) · Linear & Logistic Regression from scratch
==================================================================
Linear regression: fit the best straight line (or plane) through data by
minimizing mean-squared error. It's the "hello world" of machine learning and
the ancestor of every model in this course — a single layer with no activation,
trained by gradient descent.

Run:  python linear_regression.py
"""
import numpy as np

rng = np.random.default_rng(0)


def make_data(n=120, true_w=2.5, true_b=-1.0, noise=1.2):
    X = rng.uniform(-3, 3, size=(n, 1))
    y = true_w * X[:, 0] + true_b + noise * rng.standard_normal(n)
    return X, y


def fit_gd(X, y, lr=0.05, steps=400):
    """Fit y ≈ wX + b by gradient descent on mean-squared error."""
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(steps):
        pred = X @ w + b
        err = pred - y                      # residual
        # gradients of (1/n) Σ err² :
        gw = (2 / n) * X.T @ err
        gb = (2 / n) * err.sum()
        w -= lr * gw; b -= lr * gb
    return w, b


def normal_equation(X, y):
    """Closed-form optimum: w = (XᵀX)⁻¹ Xᵀy (with a bias column)."""
    Xb = np.c_[np.ones(len(X)), X]          # prepend a column of 1s for the bias
    theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)
    return theta[1:], theta[0]              # w, b


def r2(y, pred):
    ss_res = ((y - pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    return 1 - ss_res / ss_tot


if __name__ == "__main__":
    X, y = make_data()
    w_gd, b_gd = fit_gd(X, y)
    w_ne, b_ne = normal_equation(X, y)
    print(f"true:            w=2.500  b=-1.000")
    print(f"gradient descent w={w_gd[0]:.3f}  b={b_gd:.3f}")
    print(f"normal equation  w={w_ne[0]:.3f}  b={b_ne:.3f}")
    print(f"R^2 (fit quality, 1.0 = perfect): {r2(y, X @ w_gd + b_gd):.3f}")
    # the two methods should agree, and recover the true slope/intercept closely
    assert abs(w_gd[0] - w_ne[0]) < 0.05 and abs(b_gd - b_ne) < 0.05
    assert abs(w_gd[0] - 2.5) < 0.4
    print("OK — gradient descent and the closed-form solution agree and recover the line.")


# ============ Logistic Regression ============
"""
Track 5 (Classical ML) · Logistic Regression from scratch
=========================================================
Logistic regression turns linear regression into a classifier: push the linear
score wx+b through a sigmoid to get a probability in (0,1), and train it by
gradient descent on the cross-entropy loss (Track 3). It's a single neuron with
a sigmoid — the smallest possible neural network — and a workhorse in practice.

Run:  python logistic_regression.py
"""
import numpy as np

rng = np.random.default_rng(1)


def make_data(n=200):
    # two Gaussian blobs -> a linearly separable-ish binary problem
    n2 = n // 2
    Xa = rng.normal([-1.5, -1.0], 1.0, size=(n2, 2))
    Xb = rng.normal([1.5, 1.0], 1.0, size=(n2, 2))
    X = np.vstack([Xa, Xb]); y = np.r_[np.zeros(n2), np.ones(n2)]
    return X, y


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))


def fit(X, y, lr=0.1, steps=500):
    n, d = X.shape
    w = np.zeros(d); b = 0.0
    for _ in range(steps):
        p = sigmoid(X @ w + b)              # predicted probability of class 1
        err = p - y                         # gradient of cross-entropy is this simple!
        w -= lr * (X.T @ err) / n
        b -= lr * err.mean()
    return w, b


def accuracy(X, y, w, b):
    return ((sigmoid(X @ w + b) > 0.5).astype(int) == y).mean()


if __name__ == "__main__":
    X, y = make_data()
    w, b = fit(X, y)
    acc = accuracy(X, y, w, b)
    print(f"learned weights: {w.round(3)}, bias {b:.3f}")
    print(f"training accuracy: {acc:.3f}")
    # a probability for a clearly class-1-looking point should be high
    p = sigmoid(np.array([2.0, 1.5]) @ w + b)
    print(f"P(class 1 | a top-right point) = {p:.3f}")
    assert acc > 0.9 and p > 0.8
    print("OK — logistic regression learned a linear decision boundary with calibrated probabilities.")
